# Schema Generalization QLoRA

Train a reduced-schema QLoRA model with schema-conditioned prompts, then evaluate on a mixed seen/unseen schema test set.

In [ ]:
# Uncomment in a fresh Jupyter environment if needed.
# %pip install -q transformers datasets peft accelerate trl bitsandbytes pyyaml jsonschema

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / 'configs').exists() is False and PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
import torch

print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device =', torch.cuda.get_device_name(0))
    print('bf16_supported =', torch.cuda.is_bf16_supported())

In [ ]:
import yaml

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'train' / 'lora_phase1_qwen3b_reduced_h200fast.yaml'
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
config['experiment_name'] = 'qwen25_3b_schema_generalization_v1'
config['data']['train_path'] = 'data/processed/phase1_sft_train_reduced_schema_conditioned.jsonl'
config['data']['val_path'] = 'data/processed/phase1_sft_val_reduced_schema_conditioned.jsonl'
config['output']['save_dir'] = 'results/checkpoints/qwen25_3b_schema_generalization_v1'
config

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    'json',
    data_files={
        'train': str(PROJECT_ROOT / config['data']['train_path']),
        'validation': str(PROJECT_ROOT / config['data']['val_path']),
    },
)
dataset

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

model_name = config['model']['name']
max_seq_length = int(config['model']['max_seq_length'])
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_chat_example(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    example['text'] = text
    return example

dataset = dataset.map(format_chat_example)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False

peft_config = LoraConfig(
    r=int(config['lora']['r']),
    lora_alpha=int(config['lora']['lora_alpha']),
    lora_dropout=float(config['lora']['lora_dropout']),
    bias='none',
    task_type='CAUSAL_LM',
    target_modules='all-linear',
)

In [ ]:
training_args = TrainingArguments(
    output_dir=config['output']['save_dir'],
    learning_rate=float(config['training']['learning_rate']),
    num_train_epochs=int(config['training']['num_train_epochs']),
    per_device_train_batch_size=int(config['training']['per_device_train_batch_size']),
    per_device_eval_batch_size=int(config['training']['per_device_train_batch_size']),
    gradient_accumulation_steps=int(config['training']['gradient_accumulation_steps']),
    warmup_steps=int(config['training']['warmup_steps']),
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    bf16=use_bf16,
    fp16=not use_bf16,
    report_to='none',
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer

In [ ]:
train_result = trainer.train()
train_result

In [ ]:
output_dir = PROJECT_ROOT / config['output']['save_dir']
output_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(output_dir))
tokenizer.save_pretrained(str(output_dir))
print('saved to', output_dir)
print('exists =', output_dir.exists())
print([p.name for p in output_dir.glob('*')][:20])

## Seen / Unseen Inference Export

In [ ]:
from src.data.io import load_jsonl, dump_jsonl
from src.evaluation.metrics import try_parse_prediction_text

test_path = PROJECT_ROOT / 'data' / 'generalization' / 'phase1_test_seen_unseen_reduced.jsonl'
prediction_path = PROJECT_ROOT / 'results' / 'predictions' / 'qwen25_3b_schema_generalization_v1_test.jsonl'
prediction_path.parent.mkdir(parents=True, exist_ok=True)

test_records = load_jsonl(test_path)
len(test_records)

In [ ]:
model.eval()
predictions = []

for idx, record in enumerate(test_records):
    user_prompt = (
        f"Task: extract a structured record for {record['task_name']}.\n"
        f"Schema name: {record['schema_name']}\n"
        f"Schema definition:\n{json.dumps(__import__('src.schemas.registry', fromlist=['get_schema']).get_schema(record['schema_name']), ensure_ascii=False, separators=(',', ':'))}\n"
        'Return a JSON object only.\n'
        'Input text:\n'
        f"{record['input_text']}"
    )
    messages = [
        {'role': 'system', 'content': 'You are an information extraction model. Return only JSON that matches the requested schema. Do not add explanations or markdown.'},
        {'role': 'user', 'content': user_prompt},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            temperature=None,
            top_p=None,
            use_cache=True,
        )
    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    _, parsed = try_parse_prediction_text(generated)
    predictions.append({
        'sample_id': record['sample_id'],
        'prediction_text': generated,
        'prediction_json': parsed,
        'metadata': {
            'model_name': model_name,
            'experiment_id': config['experiment_name'],
            'schema_seen_status': record.get('schema_seen_status'),
            'schema_name': record['schema_name'],
        },
    })
    if idx % 50 == 0:
        print('processed', idx)

dump_jsonl(prediction_path, predictions)
print('saved predictions to', prediction_path)
print('exists =', prediction_path.exists())

In [ ]:
for item in predictions[:2]:
    print(item)
    print('---')